In [11]:
%pip install torch torchvision torchaudio

^C
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet152, ResNet152_Weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64
num_epochs = 5
learning_rate = 0.001

In [2]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
True
NVIDIA GeForce RTX 5060 Laptop GPU


In [4]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])



In [5]:
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)

train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

valset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_val)

val_loader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

In [6]:
model = resnet152(weights=ResNet152_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 10)
model = model.to(device)

In [7]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=learning_rate)

In [ ]:
for epoch in range(num_epochs):
  
    model.train() 
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

       
        outputs = model(images)
        loss = criterion(outputs, labels)

        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

        
        if (i + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] | Step [{i+1}/{len(train_loader)}] | Current Loss: {loss.item():.4f}")

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = (correct_train / total_train) * 100

    model.eval() 
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad(): 
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100


    print(f"\n=== Epoch [{epoch+1}/{num_epochs}] Summary ===")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")
    print(f"Val Loss:   {epoch_val_loss:.4f} | Val Acc:   {epoch_val_acc:.2f}%\n")

Epoch [1/5] | Step [100/782] | Current Loss: 0.7455
Epoch [1/5] | Step [200/782] | Current Loss: 0.5699
Epoch [1/5] | Step [300/782] | Current Loss: 0.6742
Epoch [1/5] | Step [400/782] | Current Loss: 0.6760
Epoch [1/5] | Step [500/782] | Current Loss: 0.5366
Epoch [1/5] | Step [600/782] | Current Loss: 0.8405
Epoch [1/5] | Step [700/782] | Current Loss: 0.4561

=== Epoch [1/5] Summary ===
Train Loss: 0.6985 | Train Acc: 78.52%
Val Loss:   0.5102 | Val Acc:   83.47%

Epoch [2/5] | Step [100/782] | Current Loss: 0.2751
Epoch [2/5] | Step [200/782] | Current Loss: 0.5423
Epoch [2/5] | Step [300/782] | Current Loss: 0.3613
Epoch [2/5] | Step [400/782] | Current Loss: 0.6401
Epoch [2/5] | Step [500/782] | Current Loss: 0.4368
Epoch [2/5] | Step [600/782] | Current Loss: 0.5090
Epoch [2/5] | Step [700/782] | Current Loss: 0.5101

=== Epoch [2/5] Summary ===
Train Loss: 0.4893 | Train Acc: 83.64%
Val Loss:   0.4712 | Val Acc:   84.56%

Epoch [3/5] | Step [100/782] | Current Loss: 0.2767
Epoc

        Q Why is it unnecessary (and impractical) to train ResNet-152 from scratch on small datasets? What does freezing
        most of the network tell us about the transferability of features?

        A:Training ResNet from scratch on a dataset such as CIFAR-10 is not ideal. This is due to the fact that ResNet is a  large model(relative to the dataset) with 60M parameters compared to the 50k images in CIFAR-10. Training ResNet on CIFAR from scratch will likely lead to overfitting. It is also impractical since it would require a high degree of compute. Even after freezing the backbone training the model took 20min+ on CUDA. 

        Moreover we were able to achieve more than 85% accuracy in 5 epochs. Since we froze the weights learnt from training on ImageNet it shows that the features/patterns learnt from ImageNet generalize well to other datasets such as CIFAR since it was able to accurately classify images without having to re-learn all the weights. This further shows that retraining the model is unnecessary since the ImageNet weights suffice